# Task 1: The Accuracy Metric Validation

In this section, we implement the `calculate_accuracy` function. 

**Consultant Insight:** While **Loss** is calculated using Binary Cross-Entropy (which is essential for creating a smooth gradient during backpropagation), **Accuracy** requires a hard threshold. For a binary classification with a Sigmoid output, we round the probabilistic output (e.g., $p \ge 0.5 \rightarrow 1$, else $0$) and compare it to the target labels.

In [1]:
import numpy as np

def calculate_accuracy(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """
    Calculates the accuracy of the model's predictions compared to the true labels.
    
    Why: 
    While BCE Loss gives us a continuous gradient for backpropagation, human 
    interpretation requires a concrete metric. Since our final layer uses a 
    Sigmoid activation, the output is a probability between 0 and 1. We apply 
    a threshold (0.5) to convert these probabilities into binary classifications 
    (0 for Benign, 1 for Malignant).
    """
    # 1. Apply threshold: Convert probabilities to binary predictions
    predictions = (y_pred >= 0.5).astype(int)
    
    # 2. Compare to true labels and calculate the mean of correct matches
    accuracy = np.mean(predictions == y_true)
    
    return float(accuracy)

In [2]:
# --- Validation Test ---
y_true_dummy = np.array([[1], [0], [1], [1]])
y_pred_dummy = np.array([[0.8], [0.4], [0.45], [0.9]])

acc = calculate_accuracy(y_pred_dummy, y_true_dummy)
print(f"Raw Probabilities:\n{y_pred_dummy.flatten()}")
print(f"Binary Predictions: {(y_pred_dummy >= 0.5).astype(int).flatten()}")
print(f"True Labels:        {y_true_dummy.flatten()}")
print(f"\n-> Calculated Accuracy: {acc * 100}%")

Raw Probabilities:
[0.8  0.4  0.45 0.9 ]
Binary Predictions: [1 0 0 1]
True Labels:        [1 0 1 1]

-> Calculated Accuracy: 75.0%


# Task 2: Define `update_parameters` and Parameter Math

### Mathematics of Gradient Descent

The goal of backpropagation is to calculate $dW$ and $db$, which represent the gradients (derivatives) of the Loss function with respect to the Weights and Biases. 
In other words, they tell us the *direction* and *steepness* of the error curve.

To minimize the error, we update our parameters by taking a small step in the **opposite direction** of the gradient. 

The update formulas are:
$$W = W - (\text{learning\_rate} \cdot dW)$$
$$b = b - (\text{learning\_rate} \cdot db)$$

**The Role of the Learning Rate ($\alpha$):**
- **Too large:** The steps are massive, and we might "overshoot" the minimum error, causing the loss to explode or bounce around erratically.
- **Too small:** The steps are tiny, meaning training will take forever and might get stuck in shallow local minima.

### Refactoring Strategy
Currently, `layer.py` applies `self.weights -= learning_rate * dW` directly inside its `backward` method. To adhere to standard practices and support batching, we will:
1. Ensure `backward` just stores `self.dW` and `self.db`.
2. Let the main `MultilayerPerceptron` class control when things update via an `update_parameters` method.

In [3]:
def update_parameters(network_layers, learning_rate: float):
    """
    Iterates through all layers in the network and applies the gradient descent 
    update rule to modify weights and biases.
    
    Args:
        network_layers: List of layer objects (e.g., DenseLayer instances).
        learning_rate: The step size multiplier for the gradients.
    """
    for layer in network_layers:
        # Assuming the layer's backward method has already saved dW and db as attributes.
        # W = W - (alpha * dW)
        layer.weights -= learning_rate * layer.dW
        # b = b - (alpha * db)
        layer.biases -= learning_rate * layer.db

# --- Prototype Validation ---
# Let's mock a layer object to verify the math
class MockLayer:
    def __init__(self):
        self.weights = np.array([[0.5, -0.2]])
        self.biases = np.array([[0.0, 0.0]])
        # Mock calculating gradients during backward pass
        self.dW = np.array([[0.1, -0.4]])
        self.db = np.array([[0.05, -0.1]])

layers = [MockLayer()]
learning_rate = 0.1

print(f"Original Weights:\\n{layers[0].weights}")
print(f"Original Biases:\\n{layers[0].biases}\\n")

update_parameters(layers, learning_rate)

print(f"Updated Weights:\\n{layers[0].weights}")
print(f"Updated Biases:\\n{layers[0].biases}")


Original Weights:\n[[ 0.5 -0.2]]
Original Biases:\n[[0. 0.]]\n
Updated Weights:\n[[ 0.49 -0.16]]
Updated Biases:\n[[-0.005  0.01 ]]


# Task 3: Full Epoch Training Loop Prototype

We will now stitch the forward pass, loss calculation, backward pass, and parameter update into the complete "Epoch" loop.  

An epoch is a single iteration over the entirety of the training dataset. At the end of each epoch, we will calculate our metrics on both the training set and the validation set.

*Note: For this prototype we will use dummy data and mock the forward/backward pass, so we can clearly understand the orchestration before implementing it inside the `MultilayerPerceptron` class.*

In [ ]:
import sys
import os
sys.path.append('..')

from src.loss import binary_cross_entropy, binary_cross_entropy_prime

# Let's mock a network for the training loop
class MockNetwork:
    def __init__(self):
        # We will mock the output so the loss changes
        self.epoch = 0
            
    def forward(self, X):
        # Mocking forward pass: prediction gets closer to y_true as epochs increase
        # Return an array matching the input batch size
        # [:n] means we take only the first n rows to match the input batch size(the shape of X)
        n = X.shape[0]
        if self.epoch == 0:
            return np.array([[0.1], [0.9], [0.1], [0.8]])[:n]
        elif self.epoch == 1:
            return np.array([[0.3], [0.7], [0.2], [0.85]])[:n]
        else:
            return np.array([[0.9], [0.1], [0.9], [0.9]])[:n]
            
    def backward(self, loss_grad):
        # Mock backward pass: simply takes the gradient
        pass
        
    def update_parameters(self, learning_rate):
        # Mock weight updates
        self.epoch += 1


In [7]:

# Dummy Data
X_train = np.array([[0,0], [1,1], [0,1], [1,0]])
y_train = np.array([[1], [0], [1], [1]])

X_val = np.array([[0,0], [1,1]])
y_val = np.array([[1], [0]])

# Hyperparameters
epochs = 3
learning_rate = 0.01

# Initialize our mocked network
model = MockNetwork()
history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}


In [8]:
print("Starting Training Loop...")
for epoch in range(epochs):
    # 1. Forward Pass (Training)
    y_pred_train = model.forward(X_train)

    # 2. Compute Loss and Accuracy (Training)
    train_loss = binary_cross_entropy(y_train, y_pred_train)
    train_acc = calculate_accuracy(y_pred_train, y_train)

    # 3. Backward Pass (Training)
    # Start the chain reaction by computing the gradient of the loss at the output
    loss_gradient = binary_cross_entropy_prime(y_train, y_pred_train)
    model.backward(loss_gradient)

    # 4. Update Weights
    model.update_parameters(learning_rate)

    # 5. Validation Pass (End of Epoch)
    # We do NOT run backward or update on validation data!
    y_pred_val = model.forward(
        X_val
    )  # uses updated weights because update_parameters was called
    val_loss = binary_cross_entropy(y_val, y_pred_val)
    val_acc = calculate_accuracy(y_pred_val, y_val)

    # 6. Store metrics for history
    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    # 7. Print Output (Task 4 preview)
    print(
        f"Epoch {epoch+1:02d}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
    )

Starting Training Loop...
Epoch 01/3 - loss: 1.7827 - accuracy: 0.2500 - val_loss: 1.2040 - val_acc: 0.0000
Epoch 02/3 - loss: 1.0450 - accuracy: 0.2500 - val_loss: 0.1054 - val_acc: 1.0000
Epoch 03/3 - loss: 0.1054 - accuracy: 1.0000 - val_loss: 0.1054 - val_acc: 1.0000
